# Tesla T4 — Empirical Validation Runner (H4 / H7 / H9 family)

One-paste notebook: **Runtime → Change runtime type → T4 GPU**, then **Run all**.

Clones the `t4-cuda` repo, runs `harness/empirical/run_empirical.py`, writes a
timestamped results bundle under `research/results/<ts>/`, and offers the `.tar.gz` for download.

**What it measures**
- H4 (u4/s4 INT4 LOP3): KAT bit-exactness, random-tensor allclose, ptxas registers/spills
- Fused W4A16 GEMM: max-abs-diff vs PyTorch `matmul`
- H7 / H9: math-identity reference checks (reported as *not measurable* — no CUDA kernel yet)
- Telemetry: power/clock/throttle during a stress loop
- ncu SOL: SKIPPED gracefully on Colab free tier

If the harness is not yet pushed to the repo, cell **3b** will ask you to upload it.


In [ ]:
#@title 1) Sanity check: GPU must be a Tesla T4 (or T4-class)
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda:', torch.version.cuda, '| avail:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('SM cap :', torch.cuda.get_device_capability(0) if torch.cuda.is_available() else 'NONE')


In [ ]:
#@title 2) Config
import os
REPO_URL = 'https://github.com/kridaydave/t4-cuda'  #@param {type:'string'}
BRANCH   = 'main'                                     #@param {type:'string'}
WORKDIR  = '/content/repo'                            #@param {type:'string'}
USE_TOKEN = False                                     #@param {type:'boolean'}
TOKEN = ''                                            #@param {type:'string'}
REPO_URL_AUTH = REPO_URL.replace('https://', f'https://{TOKEN}@') if (USE_TOKEN and TOKEN) else REPO_URL
print('repo :', REPO_URL)
print('branch:', BRANCH)
print('work :', WORKDIR)


In [ ]:
#@title 3) Clone repo (pinned to remote HEAD)
import os, shutil, subprocess
if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
rc = subprocess.call(['git','clone','--depth','1','-b',BRANCH,REPO_URL_AUTH,WORKDIR])
if rc != 0:
    print('git clone failed. Private repo? set USE_TOKEN+TOKEN in cell 2.')
else:
    print('cloned to', WORKDIR)
    print('HEAD:', subprocess.check_output(['git','-C',WORKDIR,'rev-parse','HEAD']).decode().strip())


### 3b) Harness presence check — upload fallback
The runner lives in `research/harness/empirical/`. If the clone does **not** contain it
(e.g. not yet pushed), this cell prompts you to upload `run_empirical.py` and
`expected.yaml` from your local `research/harness/empirical/` directory.


In [ ]:
#@title 3b) Ensure harness/empirical exists (upload if missing)
import os
EMP = os.path.join(WORKDIR, 'research', 'harness', 'empirical')
need = ['run_empirical.py', 'expected.yaml']
missing = [f for f in need if not os.path.exists(os.path.join(EMP, f))]
if not missing:
    print('harness present in clone ->', EMP)
else:
    print('MISSING in clone:', missing)
    print('A file picker will open. From your LOCAL machine, select:')
    for f in missing:
        print('  - research/harness/empirical/' + f)
    os.makedirs(EMP, exist_ok=True)
    from google.colab import files  # type: ignore
    up = files.upload()
    for name, data in up.items():
        base = os.path.basename(name)
        if base in need:
            open(os.path.join(EMP, base), 'wb').write(data)
            print('installed:', os.path.join(EMP, base))
        else:
            print('ignored (unexpected file):', name)
    still = [f for f in need if not os.path.exists(os.path.join(EMP, f))]
    assert not still, f'still missing after upload: {still}'
    print('harness ready ->', EMP)


In [ ]:
#@title 4) Install deps (Colab already has most; ensure PyYAML)
!pip -q install pyyaml
import os
RESEARCH = os.path.join(WORKDIR, 'research')
assert os.path.isdir(RESEARCH), f'No research/ dir under {WORKDIR}'
assert os.path.exists(os.path.join(RESEARCH,'harness','empirical','run_empirical.py')), 'harness missing (see cell 3b)'
print('research dir OK ->', RESEARCH)


In [ ]:
#@title 5) Execute run_empirical.py  (~10-20 min: build + tests + telemetry)
import os, subprocess
RESEARCH = os.path.join(WORKDIR, 'research')
env = dict(os.environ)
env['PYTHONPATH'] = os.path.join(RESEARCH,'src') + ':' + RESEARCH + ':' + env.get('PYTHONPATH','')
p = subprocess.run(['python3','harness/empirical/run_empirical.py'],
                   cwd=RESEARCH, env=env, text=True,
                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(p.stdout)
print('RETURN CODE:', p.returncode)


In [ ]:
#@title 6) Locate results bundle + show VERDICT.md
import os
RESROOT = os.path.join(WORKDIR, 'research', 'results')
runs = sorted([d for d in os.listdir(RESROOT) if os.path.isdir(os.path.join(RESROOT,d))])
LATEST = runs[-1]
print('runs:', runs)
print('latest:', LATEST)
vd = os.path.join(RESROOT, LATEST, 'VERDICT.md')
print('\n----- VERDICT.md -----\n')
print(open(vd).read() if os.path.exists(vd) else 'VERDICT.md missing')
tb = os.path.join(RESROOT, LATEST + '.tar.gz')
print('\ntarball:', tb if os.path.exists(tb) else 'not created')


In [ ]:
#@title 7) Download tarball
import os
try:
    from google.colab import files  # type: ignore
    if os.path.exists(tb):
        files.download(tb)
    else:
        print('No tarball found. Check cell 6 output.')
except Exception as e:
    print(f'google.colab.files not available ({e}).'
          ' Left-panel file browser can still grab research/results/*.tar.gz')


---
### After download
1. Extract the tarball inside your local `research/results/`.
2. `git add research/results/<ts>` and commit.
3. Paste `VERDICT.md` + `summary.json` back to the agent; statuses move from
   `SIMULATED_AND_FORMALLY_PROVED` to `EMPIRICALLY_VERIFIED_ON_HARDWARE` where CONFIRMED.
